In [ ]:
'''Spearman's correlation'''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import t
from itertools import permutations  

# Load the data
df = pd.read_csv('whole genome.csv')
df.columns = [col.strip() for col in df.columns]

groups = ["All SSR", "P-SSR", "C-SSR", "I-SSR"]
variables = ["Number", "Length", "Abundance", "Density"]

y_labels = {
    'Number': 'Number (loci)',
    'Length': 'Length (bp)',
    'Abundance': 'Abundance (loci/Mb)',
    'Density': 'Density (bp/Mb)'
}

def calculate_exact_spearman_p(x, y):
    obs_rho, _ = stats.spearmanr(x, y)
    y_perms = list(permutations(y))
    count_extreme = 0
    
    for y_p in y_perms:
        r_sim, _ = stats.spearmanr(x, y_p)
        
        if abs(r_sim) >= abs(obs_rho) - 1e-10: # Adding a small tolerance to account for floating-point precision
            count_extreme += 1
            
    p_exact = count_extreme / len(y_perms)
    return obs_rho, p_exact

# Set up figure
fig, axes = plt.subplots(4, 4, figsize=(16, 16), dpi=600)
colors = {'All SSR': '#ef8297', 'P-SSR': '#FADFA1', 'C-SSR': '#A1D6B2', 'I-SSR': '#7EACB5'}

variable_columns = {
    'All SSR': {'Number': 'Total repeat number', 'Length': 'Total repeat length', 'Abundance': 'Total repeat abundance', 'Density': 'Total repeat density'},
    'P-SSR': {'Number': 'Number of P-SSR', 'Length': 'Length of P-SSR', 'Abundance': 'Abundance of P-SSR', 'Density': 'Density of P-SSR'},
    'C-SSR': {'Number': 'Number of C-SSR', 'Length': 'Length of C-SSR', 'Abundance': 'Abundance of C-SSR', 'Density': 'Density of C-SSR'},
    'I-SSR': {'Number': 'Number of I-SSR', 'Length': 'Length of I-SSR', 'Abundance': 'Abundance of I-SSR', 'Density': 'Density of I-SSR'}
}

# Column Check loop 
for group in groups:
    for variable in variables:
        col_name = variable_columns[group][variable]
        if col_name not in df.columns:
            for actual_col in df.columns:
                if col_name.replace(" ", "").lower() in actual_col.replace(" ", "").lower():
                    variable_columns[group][variable] = actual_col
                    break

# Plotting loop
for i, variable in enumerate(variables):
    for j, group in enumerate(groups):
        ax = axes[i, j]

        try:
            y_column = variable_columns[group][variable]
            x_column = 'Total valid genome length'

            x = df[x_column].values
            y = df[y_column].values

            valid = ~(np.isnan(x) | np.isnan(y))
            x = x[valid]
            y = y[valid]

            ax.scatter(x, y, color='black', s=30)

            if len(x) > 1:
                
                rho, p_val = calculate_exact_spearman_p(x, y)

                # Linear Regression 
                slope, intercept, _, _, _ = stats.linregress(x, y)
                x_line = np.linspace(min(x), max(x), 100)
                y_line = intercept + slope * x_line

                # CI Calculation
                n = len(x)
                t_value = t.ppf(0.975, df=n - 2)
                y_fit = slope * x + intercept
                residuals = y - y_fit
                s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
                ci = t_value * s_err * np.sqrt(1/n + (x_line - np.mean(x))**2 / np.sum((x - np.mean(x))**2))

                ax.plot(x_line, y_line, color=colors[group])
                ax.fill_between(x_line, y_line - ci, y_line + ci, color=colors[group], alpha=0.3)

                
                significance = " *" if p_val < 0.05 else ""
                ax.text(0.05, 0.95, f"Rho = {rho:.2f}, p = {p_val:.3f}{significance}",
                        transform=ax.transAxes, fontsize=14, verticalalignment='top')

        except Exception as e:
            print(f"Error plotting {group} {variable}: {e}")

        if i == 0: ax.set_title(group,fontsize=16, fontweight="bold", pad=15)
        if j == 0: ax.set_ylabel(y_labels[variable],fontsize=16, fontweight="bold", labelpad=15)
        if i == 3: ax.set_xlabel('Genome Size',fontsize=16, fontweight="bold", labelpad=15)

        ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
        ax.tick_params(axis='both', labelsize=12)
fig.tight_layout()

plt.show()